# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WingManOO7/ML-INTER/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup — the frozen policy from ML-09

Nothing is re-tuned in this notebook. The policy was frozen after the ML-09 validation:

- **Score:** P(decline) × lost weekly impressions, where P comes from logistic regression on the five
  ML-04 contract features, trained on the spike-robust label `declined_robust`.
- **Routes:** the error analysis, promoted to policy. A page with a warning sign gets a different action
  instead of a content refresh (rules in `search_queue.ROUTES`, unit-tested).
- **Queue date:** **2026-05-11**, the last development date. The model trains only on dates whose
  outcomes ended before it.

In [1]:
import os, sys, json, subprocess, importlib.util, urllib.request, warnings
from pathlib import Path
warnings.filterwarnings('ignore', category=FutureWarning)

if importlib.util.find_spec('duckdb') is None:      # Colab ships duckdb; this covers fresh local envs
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'duckdb'])

import numpy as np
import pandas as pd
import sklearn

# the shared module: from the repo when run inside it, else fetched from the repo (Colab opened from GitHub)
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'work' / 'notebooks').is_dir()), None)
if ROOT is not None:
    sys.path.insert(0, str(ROOT / 'work' / 'scripts'))
else:
    ROOT = Path.cwd()
    urllib.request.urlretrieve('https://raw.githubusercontent.com/WingManOO7/ML-INTER/main/work/scripts/search_queue.py',
                               'search_queue.py')
import search_queue as sq

# Token order: env var -> Colab Secret -> prompt (last resort). Never paste a token into a cell.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
if not HF_TOKEN:
    import getpass
    HF_TOKEN = getpass.getpass('Hugging Face READ token: ')

OUT_DIR = ROOT / 'work' / 'outputs'
CACHE = OUT_DIR / 'cache'            # parquet frames, gitignored: reruns skip the warehouse scans
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEV_DATES = sq.decision_dates('2026-01-05', '2026-05-11')     # every second Monday; outcomes end 2026-05-24
DEV_MONTHS = sq.months_needed(DEV_DATES)
assert '2026-06' not in DEV_MONTHS, 'June 2026 is the sealed month: development never opens it'

con = sq.open_warehouse(DEV_MONTHS, HF_TOKEN)
frames = sq.build_frames(con, DEV_DATES, cache_dir=CACHE)
dev = frames[frames['label_ok']].reset_index(drop=True)
SEED = 42

print(f'{len(DEV_DATES)} decision dates {DEV_DATES[0].date()} -> {DEV_DATES[-1].date()} | months {DEV_MONTHS[0]} -> {DEV_MONTHS[-1]}')
print(f'{len(frames):,} page-date rows; {len(dev):,} with a complete outcome window; '
      f'{dev.client_hash_id.nunique()} clients | pandas {pd.__version__}, scikit-learn {sklearn.__version__}')

10 decision dates 2026-01-05 -> 2026-05-11 | months 2025-12 -> 2026-05
659,482 page-date rows; 654,186 with a complete outcome window; 50 clients | pandas 2.3.3, scikit-learn 1.7.2


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

| Action | Reason code | When | What the editor does |
|---|---|---|---|
| `TECHNICAL_CHECK` | `COLLAPSE_TO_NEAR_ZERO` | week 2 has < 5% of week-1 impressions | check indexing, redirects, robots, canonical: a refresh can't fix a page search can't see |
| `SITE_LEVEL_CHECK` | `SITE_WIDE_SLIDE` | the whole site lost ≥ 20% from week 1 to week 2 | look at the site first (tracking, migration, a search update): many pages, one cause |
| `MONITOR` | `SPIKE_FADE` | one day held > 50% of the feature window | wait a week: this is likely a burst ending, not decay |
| `MONITOR` | `NEW_PAGE_SETTLING` | page younger than 30 days | wait: new pages settle after launch exposure |
| `REFRESH_REVIEW` | `SLIDING_WITH_DEMAND` | none of the above | review the content: intent, freshness, coverage, snippet |

Every page also carries a **confidence band** from its P(decline). The band thresholds are read off the
rolling backtest: the precision each band actually reached at the top of the queue (table below).

In [2]:
SCORE_DATE = DEV_DATES[-1]
train = dev[dev['decision_date'] < SCORE_DATE]
today = frames[frames['decision_date'] == SCORE_DATE].copy()
sq.assert_time_forward(train, today)
model = sq.fit_policy_model(train)

today['p_decline'] = model.predict_proba(today[sq.CONTRACT_FEATURES])[:, 1]
today['lost_weekly_impressions'] = sq.lost_impressions(today)
today['expected_loss'] = sq.queue_score(today['p_decline'], today)
today = today.join(sq.route(today))

# confidence bands, calibrated on the rolling backtest (top-50 picks of every date x fold cell)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
make_lr = lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
bt = sq.rolling_backtest(frames, {'lr5': (sq.CONTRACT_FEATURES, make_lr)})
bt['score'] = sq.queue_score(bt['p_lr5'], bt)
bt['rank'] = bt.groupby(['decision_date', 'fold'])['score'].rank(ascending=False, method='first')
picks = bt[bt['rank'] <= 50]
BANDS = [0, 0.4, 0.6, 1.01]
band_names = ['low (P < 0.4)', 'medium (0.4-0.6)', 'high (P >= 0.6)']
calibration = picks.groupby(pd.cut(picks['p_lr5'], BANDS, labels=band_names, right=False), observed=True)['declined_robust'] \
                   .agg(picks='size', observed_precision='mean')
print('Backtest calibration of the confidence bands (top-50 picks, 40 date x fold cells):')
print(calibration.round(3).to_string(), '\n')
today['confidence'] = pd.cut(today['p_decline'], BANDS, labels=['low', 'medium', 'high'], right=False).astype(str)

queue = today[today['action'] != 'NO_ACTION'].sort_values('expected_loss', ascending=False).reset_index(drop=True)
queue.insert(0, 'rank', np.arange(1, len(queue) + 1))
print(f"{SCORE_DATE.date()}: {len(today):,} pages scored, {len(queue):,} with a drop before T enter the queue\n")
view = queue.head(20)[['rank', 'action', 'reason_code', 'confidence', 'p_decline', 'lost_weekly_impressions',
                       'expected_loss', 'imp_pre', 'pos_pre', 'age_days_at_T']]
view.round({'p_decline': 2, 'expected_loss': 0, 'pos_pre': 1}).set_index('rank')

Backtest calibration of the confidence bands (top-50 picks, 40 date x fold cells):
                  picks  observed_precision
p_lr5                                      
low (P < 0.4)       417               0.458
medium (0.4-0.6)    626               0.636
high (P >= 0.6)     957               0.834 

2026-05-11: 79,578 pages scored, 40,316 with a drop before T enter the queue



,action,reason_code,confidence,p_decline,lost_weekly_impressions,expected_loss,imp_pre,pos_pre,age_days_at_T
rank,,,,,,,,,
1,REFRESH_REVIEW,SLIDING_WITH_DEMAND,medium,0.58,36153.0,21004.0,68829.0,6.6,508
2,REFRESH_REVIEW,SLIDING_WITH_DEMAND,high,0.80,22464.0,17942.0,29866.0,7.5,392
3,REFRESH_REVIEW,SLIDING_WITH_DEMAND,high,0.93,13543.0,12644.0,15131.0,7.1,194
4,TECHNICAL_CHECK,COLLAPSE_TO_NEAR_ZERO,high,0.95,11457.0,10865.0,12569.0,10.5,200
5,REFRESH_REVIEW,SLIDING_WITH_DEMAND,high,0.70,14334.0,9992.0,22700.0,8.8,392
6,REFRESH_REVIEW,SLIDING_WITH_DEMAND,high,0.74,12654.0,9357.0,20056.0,8.9,68
7,TECHNICAL_CHECK,COLLAPSE_TO_NEAR_ZERO,high,0.97,9295.0,9028.0,9707.0,3.3,508
8,REFRESH_REVIEW,SLIDING_WITH_DEMAND,medium,0.50,18076.0,8949.0,56502.0,4.6,55
9,REFRESH_REVIEW,SLIDING_WITH_DEMAND,high,0.81,10350.0,8404.0,13918.0,4.0,118


**The bands mean what they say.** In the backtest, the top-50 picks in the *high* band (P ≥ 0.6) went on to decline **83%** of the time, *medium* 64% and *low* 46%, against development base rates of 0.14–0.57. Each step is well apart from the next, so an editor can rely on the band.

**The 2026-05-11 queue.** 79,578 pages were scored; the 40,316 that lost impressions before T enter the queue, ordered by expected loss. The top 20 are large pages (up to 79,348 impressions in two weeks) that lost thousands of impressions per week, and **most carry a high or medium band**.

The score balances the two parts. Rank 1 lost 36,153 weekly impressions at a medium P of 0.58. Rank 12 lost 16,226 at P 0.42, but rank 13 (16,838 lost, P 0.38) sits right below it, because a large loss with a low chance of persisting is worth less than a slightly smaller loss that is likely to stick. Routes already change the action for some rows: three collapses go to `TECHNICAL_CHECK`, two new pages to `MONITOR` and two to `SITE_LEVEL_CHECK`.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Who and what.** A content editor, or the person who plans editor time, opens the queue at each decision
date (every second Monday) and works it from the top: about **50 pages per cycle**, the capacity
fixed in ML-03. The queue says *where to look first and why*. It never says what to change.

**Valid for:**
- pages with **≥ 100 Search Console impressions** in the 14 days before T, tracked for all 14 days, and
  with a valid position;
- a **two-week horizon**: "will this page's visibility over the next two weeks sit more than 20% below
  the last two (ignoring each window's single biggest day)?";
- sites in the same pseudonymized FlyRank portfolio and the same kind of period as January–May 2026.

**Not valid for:**
- **pages below the floor.** Absence from the queue is *not* evidence that a page is healthy.
- **predicting fresh drops.** The ranker is strongest at telling which *visible* drops persist (ML-08
  §4d). It is not an early warning for pages that look fine today.
- **causal claims.** Nothing here shows that refreshing a page brings its visibility back; that needs an
  experiment.
- **weeks like 2026-04-27**, when the whole portfolio moved. There the label mostly measures the market,
  not the page, which is what the monitoring triggers in section 4 are for.

In [3]:
all_today = con.sql(f'''
    SELECT COUNT(DISTINCT content_hash_id) AS tracked_pages,
           SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE) AS impressions
    FROM daily WHERE report_date >= DATE '{(SCORE_DATE - pd.Timedelta(days=14)).date()}' AND report_date < DATE '{SCORE_DATE.date()}'
''').df().iloc[0]
coverage = {
    'pages tracked in the 14 days before T': int(all_today['tracked_pages']),
    'pages eligible for scoring': len(today),
    'share of tracked pages that are eligible': len(today) / all_today['tracked_pages'],
    'share of all impressions held by eligible pages': today['imp_pre'].sum() / all_today['impressions'],
    'pages in the queue (dropped before T)': len(queue),
}
for k, v in coverage.items():
    print(f'{k:>50}: {v:,.3f}' if isinstance(v, float) else f'{k:>50}: {v:,}')

             pages tracked in the 14 days before T: 372,388
                        pages eligible for scoring: 79,578
          share of tracked pages that are eligible: 0.214
   share of all impressions held by eligible pages: 0.974
             pages in the queue (dropped before T): 40,316


**What the floor leaves out, and what it keeps.** 372,388 pages appeared in the warehouse in the 14 days before 2026-05-11; 79,578 (**21.4%**) pass the floor. Those pages hold **97.4% of all impressions**. The floor drops four out of five pages, but they are the pages with almost no search visibility, so the queue covers nearly all of the visibility there is to protect. The other 78.6% of pages are simply *unscored*. They are not "fine".

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Before acting on any row, a person checks:**
1. **Is the drop real?** Open Search Console for the page: is the loss spread over many days, or one
   bad day? Is it tracking (a missing property, a filter) rather than search?
2. **Is it the page or the site?** Compare with the site's other pages. If many fell together, it is a
   `SITE_LEVEL_CHECK`, whatever the row says.
3. **Did demand move?** A seasonal topic or a news burst ending is not decay.
4. **Did a sibling page take over?** Consolidation, where another URL on the same site now ranks for the
   query, means *merge or redirect*, not *refresh*.
5. **Is the page still meant to exist?** A product that was retired, or a page that was de-indexed on
   purpose, is not a refresh target.

**The no-go list:**
- ✗ No automatic edits, rewrites, deletions, redirects or noindex from this queue. Every action is a
  human decision.
- ✗ No site-level conclusions from a page queue, and no ranking of clients or editors by queue numbers.
- ✗ No promise of recovery. The score ranks attention; it doesn't forecast results of an edit.
- ✗ No use of the pseudonymous IDs to re-identify clients or pages, and no raw queries or URLs in
  outputs.
- ✗ No scoring of pages below the 100-impression floor, and no reading of "not in the queue" as "healthy".
- ✗ No use in a week when a monitoring trigger has paused the queue (section 4).

In [4]:
# what a reviewer sees: the highest-scoring example of each route (numbers only, no identifiers)
cols = ['rank', 'action', 'reason_code', 'confidence', 'imp_wk1', 'imp_wk2', 'peak_day_share_pre',
        'site_momentum', 'age_days_at_T', 'pos_pre', 'expected_loss']
examples = queue.groupby('reason_code', sort=False).head(1)[cols]
mix_top50 = queue.head(50)['reason_code'].value_counts()
print('Route mix in the top 50:', mix_top50.to_dict(), '\n')
examples.round({'peak_day_share_pre': 2, 'site_momentum': 2, 'pos_pre': 1, 'expected_loss': 0}).set_index('rank')

Route mix in the top 50: {'SLIDING_WITH_DEMAND': 38, 'SITE_WIDE_SLIDE': 6, 'COLLAPSE_TO_NEAR_ZERO': 4, 'NEW_PAGE_SETTLING': 2} 



,action,reason_code,confidence,imp_wk1,imp_wk2,peak_day_share_pre,site_momentum,age_days_at_T,pos_pre,expected_loss
rank,,,,,,,,,,
1,REFRESH_REVIEW,SLIDING_WITH_DEMAND,medium,52491.0,16338.0,0.45,-0.03,508,6.6,21004.0
4,TECHNICAL_CHECK,COLLAPSE_TO_NEAR_ZERO,high,12013.0,556.0,0.21,-0.13,200,10.5,10865.0
11,MONITOR,NEW_PAGE_SETTLING,high,13424.0,3464.0,0.23,0.37,28,8.1,7079.0
16,SITE_LEVEL_CHECK,SITE_WIDE_SLIDE,medium,29043.0,17400.0,0.11,-0.27,195,7.8,5116.0
59,MONITOR,SPIKE_FADE,high,3362.0,177.0,0.52,0.00,110,9.2,3031.0


**What the reviewer sees, per route** (2026-05-11; the first page of each route in the queue):
- **Rank 1, `REFRESH_REVIEW`:** 52,491 → 16,338 weekly impressions, steady site, no spike (peak day 0.45), position 6.6. A real slide on a large page.
- **Rank 4, `TECHNICAL_CHECK`:** 12,013 → 556. A 95% collapse in one week usually means indexing or a redirect, not stale content.
- **Rank 11, `MONITOR` (`NEW_PAGE_SETTLING`):** a 28-day-old page falling back from its launch exposure.
- **Rank 16, `SITE_LEVEL_CHECK`:** the page slid while its whole site lost 24% (`site_momentum` −0.27).
- **Rank 59, `MONITOR` (`SPIKE_FADE`):** one day held 52% of its feature window, so the "drop" is mostly that day ending.

In the top 50, **38** rows are content-refresh candidates, and **12** are routed elsewhere before anyone spends an hour rewriting them.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Two kinds of check. **Pre-release** checks need no labels and run the moment a new queue is built.
**Post-label** checks run two weeks later, when the outcomes are in.

| # | Check | When | Trigger | Action |
|---|---|---|---|---|
| T1 | feature drift: PSI of each feature vs the **previous cycle** (`age_days_at_T` left out) | pre-release | any PSI > 0.25 | hold the queue for a human look before release |
| T2 | portfolio event: share of the top 50 routed `SITE_LEVEL_CHECK` | pre-release | > 0.5 | pause page-level refresh work; investigate sites first |
| T3 | base-rate shift vs the previous cycle | post-label | moved > 0.10 | annotate the cycle: judge it by lift over base rate, not raw precision |
| T4 | the model loses to the frozen rule (precision@50) | post-label | two cycles in a row | fall back to the ML-07 rule while investigating |

**Two design choices, both learned from a first version of this table:**
- **Compare with the previous cycle, not with all history.** The panel keeps growing (27 → 45 clients),
  so "all history" drifts every cycle.
- **Leave `age_days_at_T` out of T1.** Every page is 14 days older at each cycle *by construction*. A
  first version that included it, against all history, fired on 5 of 8 dates: an alarm nobody would read.

**Retrain cadence:** refit at every decision date on all labelled history, exactly as the backtest does.
The replay below runs the triggers over the development history.

In [5]:
rows = []
dates = [pd.Timestamp(d) for d in sorted(dev['decision_date'].unique())]
cells = sq.cell_metrics(bt, {'model': 'score', 'rule': sq.rule_score})
p50 = cells.pivot_table(index='decision_date', columns='policy', values='P@50')
loses = (p50['model'] < p50['rule'])
DRIFT_FEATURES = [c for c in sq.CONTRACT_FEATURES if c != 'age_days_at_T']     # age moves +14 days per cycle by design
base_by_date = dev.groupby('decision_date')['declined_robust'].mean()
for i, T in enumerate(dates[2:], start=2):
    prev, now = frames[frames['decision_date'] == dates[i - 1]], frames[frames['decision_date'] == T]
    psis = {c: sq.psi(prev[c], now[c]) for c in DRIFT_FEATURES}
    top = bt[(bt['decision_date'] == T)].sort_values('score', ascending=False).groupby('fold').head(50)
    site_share = (sq.route(top)['action'] == 'SITE_LEVEL_CHECK').mean()
    two_losses = bool(loses.get(T, False) and loses.get(dates[i - 1], False))
    rows.append({'decision_date': T.date(), 'T1 max PSI': max(psis.values()), 'T1 feature': max(psis, key=psis.get),
                 'T2 site share top50': site_share, 'T3 base-rate move': base_by_date[T] - base_by_date[dates[i - 1]],
                 'T4 model < rule (2 in a row)': two_losses})
replay = pd.DataFrame(rows).set_index('decision_date')
replay['fires'] = [', '.join(t for t, hit in [('T1', r['T1 max PSI'] > 0.25), ('T2', r['T2 site share top50'] > 0.5),
                                              ('T3', abs(r['T3 base-rate move']) > 0.10), ('T4', r['T4 model < rule (2 in a row)'])] if hit) or '—'
                   for _, r in replay.iterrows()]
replay.round(3)

,T1 max PSI,T1 feature,T2 site share top50,T3 base-rate move,T4 model < rule (2 in a row),fires
decision_date,,,,,,
2026-02-02,0.072,momentum_pre,0.184,0.080,False,—
2026-02-16,0.073,momentum_pre,0.024,-0.037,False,—
2026-03-02,0.010,log_imp_pre,0.000,0.129,False,T3
2026-03-16,0.089,momentum_pre,0.208,0.026,False,—
2026-03-30,0.084,momentum_pre,0.064,0.061,False,—
2026-04-13,0.053,momentum_pre,0.136,0.031,False,—
2026-04-27,0.482,momentum_pre,0.604,0.147,False,"T1, T2, T3"
2026-05-11,0.637,momentum_pre,0.168,-0.229,False,"T1, T3"


**The replay behaves like a monitor should: quiet in normal weeks, loud at the portfolio event.**
- **2026-04-27:** **T1 and T2 fire before release.** `momentum_pre` shifted against the previous cycle (PSI 0.48), and 60% of the top 50 routed to `SITE_LEVEL_CHECK`. That is the week the whole portfolio lost 18% of its impressions: the queue would have been held and the sites checked first. **The pre-release checks saw it coming from week-2 data, before any label existed.**
- **2026-05-11:** T1 fires again (PSI 0.64) as the portfolio moved back. T3 annotates a base-rate fall of 0.23.
- **2026-03-02:** only the T3 annotation fires (base rate +0.13).
- **T4** never fired: the model never lost to the frozen rule two cycles in a row.

That is 3 alerting cycles out of 8, and the two stop-level checks fired only at the portfolio event and the cycle after it.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

- `work/outputs/action_queue.csv`: the full queue with pseudonymous IDs. It is **gitignored**, and the
  notebook regenerates it on every run.
- `work/outputs/w07_playbook_receipts.json`: the numbers above, with no identifiers. Committed.

In [6]:
export_cols = ['rank', 'decision_date', 'client_hash_id', 'content_hash_id', 'action', 'reason_code', 'confidence',
               'p_decline', 'lost_weekly_impressions', 'expected_loss', 'imp_pre', 'pos_pre', 'age_days_at_T']
queue[export_cols].to_csv(OUT_DIR / 'action_queue.csv', index=False)
assert not sq.OUTCOME_COLUMNS & set(export_cols), 'the queue must not carry outcome columns'

receipts = {
    'notebook': 'work/notebooks/w07_action_playbook.ipynb', 'score_date': str(SCORE_DATE.date()),
    'policy': 'P(declined_robust | 5 contract features, logistic regression) x lost weekly impressions',
    'pages_scored': len(today), 'queue_rows': len(queue),
    'route_counts_all': queue['reason_code'].value_counts().to_dict(),
    'route_counts_top50': mix_top50.to_dict(),
    'confidence_calibration': calibration.round(4).reset_index().astype(str).to_dict(orient='records'),
    'coverage': {k: (round(float(v), 4) if isinstance(v, float) else v) for k, v in coverage.items()},
    'trigger_replay': replay.reset_index().astype(str).to_dict(orient='records'),
    'top20_no_ids': view.round(4).astype(str).to_dict(orient='records'),
}
(OUT_DIR / 'w07_playbook_receipts.json').write_text(json.dumps(receipts, indent=2, default=lambda o: o.item() if hasattr(o, 'item') else str(o)))
print('wrote', (OUT_DIR / 'action_queue.csv').relative_to(ROOT), f'({len(queue):,} rows, gitignored) and',
      (OUT_DIR / 'w07_playbook_receipts.json').relative_to(ROOT))

wrote work\outputs\action_queue.csv (40,316 rows, gitignored) and work\outputs\w07_playbook_receipts.json


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.